In [ ]:
from pathlib import Path

cwd = Path.cwd()

QnA = r"C:\All University Materials\Project\ICT304-project\ai\data_extractor\data\raw"

In [ ]:
def kwords(*stuff):
    return stuff

value = [('meow','meow')]
kwords(*value)

In [ ]:
import re

def search_txt_files(dir_path : str, files=None):
    
    if files is None:
        files = list()
    
    current_dir = Path(dir_path)
    
    for item in current_dir.iterdir():
        
        if re.search(r'\.txt', item.name):
            files.append(item)
        elif item.is_dir():
            search_txt_files(str(item), files)
            
    return files

In [ ]:
import json

with open('./test.json', 'w') as f:
    json.dump({'hello' : 1}, f)

In [ ]:
import re

def create_qna_pairs(split_txt_list : list):
        
    pair_storage = []
    
    i = 0
    while i < len(split_txt_list) - 3:
        
        Q = split_txt_list[i].strip()
        A = split_txt_list[i + 2].strip()
            
        is_Q = re.search(r"^([Qq].?)$", Q) is not None
        is_A = re.search(r"^([Aa].?)$", A) is not None
        
        print(is_Q and is_A)
        
        if is_Q and is_A:
            
            question = split_txt_list[i + 1].strip()
            answer = split_txt_list[i + 3].strip()
            
            pair_storage.append({'Question': question, 'Answer': answer})
            
            i += 4
        else:
            i += 2
            
        
            
    # old logic
    
    """while len(split_txt_list) >= 4:
        
        pair = split_txt_list[:4:]
        
        A = pair[1].strip()
        Q = pair[3].strip()
        
        if len(A) > 5 and len(Q) > 5:
            pair_storage.append({'Question' : A, 'Answer' : Q})
        
        split_txt_list = split_txt_list[4::]"""
        
    return pair_storage

In [ ]:
[1,2,3,4][:3]

In [ ]:

def extract_qna_pairs(txt_file_paths: list[Path]):
    
    text = []
    
    for txt_file in txt_file_paths:
        with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
            raw_text = f.read()

            # split on Q and A tags
            split_text = re.split(r'(\n+[AaQq].?\n)', raw_text)
            
            # remove the preamble from shri's top text
            while split_text and re.search(r'^([AaQq].?)$', split_text[0].strip()) is None:
                split_text.pop(0) 
                
            text = text + create_qna_pairs(split_text)
                
    return text


In [ ]:
pairs = extract_qna_pairs(search_txt_files(QnA))

In [ ]:
pairs

In [ ]:
import json

def create_json(pair_list):
    
    with open('data.json', 'w') as file:
        json.dump(pair_list, file, indent=4)
        
create_json(pairs)

In [ ]:
import zipfile
from pathlib import Path

cwd = Path.cwd()
data_dir = cwd.parent / 'data'
ICT_283 = data_dir / 'raw' / 'ICT283_contents'

with zipfile.ZipFile(ICT_283 / 'ICT283_content.zip') as zip:
    
    zip.extractall(ICT_283)




In [ ]:
# files contained in 283 stuff
# word doc
# power point
# txt files
# c files
# cpp files
# h files

SLIDES

In [ ]:
from pptx import Presentation
import os
import comtypes.client


# gemini made this code, couldn't be arsed to figure it out
def convert_ppt_to_pptx(ppt_path):
    # Absolute paths are required for COM objects
    abs_path = os.path.abspath(ppt_path)
    output_path = abs_path + "x"  # Changes .ppt to .pptx

    # Initialize PowerPoint
    ppt_app = comtypes.client.CreateObject("Powerpoint.Application")
    
    # Optional: Keep it invisible (0) or show it (1)
    ppt_app.Visible = 0

    try:
        # Open the legacy file
        deck = ppt_app.Presentations.Open(abs_path)
        
        # Save as .pptx (FileFormat 24)
        # Reference: https://learn.microsoft.com/en-us/office/vba/api/powerpoint.ppsaveasfileformat
        deck.SaveAs(output_path, 24) 
        deck.Close()
        print(f"Success! Created: {output_path}")
        return output_path
    except Exception as e:
        print(f"Error: {e}")
    finally:
        ppt_app.Quit()
        
path = r"C:\All University Materials\Project\ICT304-project\ai\ai_tools\data_extractor\data\raw\ICT283_contents\Topic03\Content\Lec-07.ppt"
new_pptx_path = convert_ppt_to_pptx(path)

EXTRACTORS

In [ ]:
import ollama
import subprocess
import time

subprocess.run(['taskkill','/f','/im', 'ollama.exe'])

server = subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.PIPE,
                 stderr=subprocess.PIPE)

time.sleep(1)

print('ready')


In [ ]:
from abc import ABC, abstractmethod

from pathlib import Path
import shutil
import zipfile
import comtypes.client
from PIL import Image
import io
import os
import tempfile
import subprocess
import ollama
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE

from unit_extractor.base import ExtractorBase

In [ ]:
extractor

In [ ]:
"""
PdfExtractor — extracts text and images from PDFs, one dict per page.

Add to your existing imports:
    import fitz   # PyMuPDF — install with `pip install pymupdf`

Assumes the following are already imported in your file from the PPT work:
    io, os, tempfile
    from pathlib import Path
    from PIL import Image
"""
import fitz

import ollama

def img_to_text(img, model='qwen3.5:397b-cloud'):
    response = ollama.chat(model=model,
                messages=[{'role': 'system', 'content': 'You are tasked with describing images in as much detail as possible. For all images, describe them in depth, especially any text available.'},
                          {'role': 'user', 'content': 'Describe this image in detail. Limit your response to less than 200 words or less. , focusing on labeled entities and their relationships. Do not describe colors, fonts, or visual style unless they convey meaning.', 'images': img}],
                think=True,
                options={'temperature':0})
    
    return response.message.content

class PdfExtractor(ExtractorBase):
    """
    Extracts a PDF page-by-page. For each page, pulls the text layer and
    any embedded images, runs the images through the vision model (via
    kwargs['img_to_txt']), and concatenates everything into a single
    per-page "big string" — same shape as PptExtractor's output.
    """

    def extract(self, path: str | Path = None) -> list:
        path = Path(path).absolute()

        doc = fitz.open(path)
        extracted_pages = []

        try:
            for i, page in enumerate(doc):
                text = self._get_text(page)
                images = self._get_images(doc, page, i + 1)

                big_string = (
                    f"--- PAGE {i+1} ---\n"
                    f"CONTENT:\n{text}\n"
                    f"IMAGES:\n{images} --- "
                )

                extracted_pages.append({
                    "id": str(path),
                    "text": big_string,
                    "metadata": {
                        "filename": path.name,
                        "extension": path.suffix.lstrip('.').lower(),
                        "topic": path.parent.name,
                        "path": str(path),
                    }
                })
        finally:
            doc.close()

        return extracted_pages

    def _optimize_image(self, blob: bytes, max_size=(512, 512), min_dim=32) -> bytes | None:
        """
        Standardizes images to JPEG, resizes them, and filters out tiny ones.
        Duplicated from PptExtractor — could be factored up into ExtractorBase
        later if you want to dedupe.
        """
        try:
            with Image.open(io.BytesIO(blob)) as img:
                if img.width < min_dim or img.height < min_dim:
                    print(f"    [!] Skipping tiny image: {img.width}x{img.height}")
                    return None

                if img.mode != 'RGB':
                    img = img.convert('RGB')

                img.thumbnail(max_size, Image.Resampling.LANCZOS)

                output = io.BytesIO()
                img.save(output, format='JPEG', quality=85)
                return output.getvalue()
        except Exception as e:
            print(f"    [!] Error optimizing image: {e}")
            return None

    def _get_text(self, page) -> str:
        """
        Retrieves the text layer from a PDF page in natural reading order.
        Returns empty string for scanned PDFs (no text layer).
        """
        return page.get_text("text")

    def _get_images(self, doc, page, page_num) -> str:
        """Extracts embedded images from the page and runs each through img_to_txt."""
        images = ""

        # page.get_images(full=True) returns tuples describing each image.
        # The first element is the xref (PDF's internal reference number).
        for img_info in page.get_images(full=True):
            xref = img_info[0]

            try:
                img_dict = doc.extract_image(xref)
            except Exception as e:
                print(f"    [!] Couldn't extract image xref={xref} on page {page_num}: {e}")
                continue

            ext = img_dict.get("ext", "").lower()
            blob = img_dict.get("image")

            if ext not in ['png', 'jpg', 'jpeg', 'bmp', 'tiff']:
                continue

            print(f"Processing ID: p{page_num}_{xref} | Ext: {ext}")

            optimized_blob = self._optimize_image(blob)
            if optimized_blob is None:
                continue

            try:
                extracted_text = self.kwargs['img_to_txt'](img=[optimized_blob])
                images += f"ID: p{page_num}_{xref}\nIMG_TO_TXT:\n{extracted_text}\n"
            except Exception as e:
                print(f"    [!] Failed to process image with model: {e}")

        return images

test_path_2 = r"C:\All University Materials\Project\ICT304-project\ai\ai_tools\data_extractor\data\raw\ICT283_contents\Topic10\Content\coding standards and rules\JSF-AV-C++rules.pdf"
extractor = PdfExtractor(img_to_txt=img_to_text).extract(test_path_2)

In [ ]:
ollama.list().models

In [ ]:
from queue import LifoQueue, Queue



output = Queue()

file_path = r"C:\All University Materials\Project\ICT304-project\ai\ai_tools\data_extractor\data\raw"

def extract_text(directory : str | Path, **fnc_dict):
    
    stack = LifoQueue()
    handlers = fnc_dict
    document = []
    
    for path in Path(directory).iterdir():
        stack.put(path)
    
    while not stack.empty():
        
        current = stack.get()
        
        if isinstance(current, dict):
            document.append(current)
            continue
        
        current_as_path = Path(current)
        
        if current_as_path.is_dir():
            
            for path in current_as_path.iterdir():
                stack.put(path)
        else:
            file_type = current_as_path.suffix.lstrip('.').lower()
            file_function = handlers.get(file_type, None)
            
            if not file_function:
                continue
            else:
                
                for extracted in file_function.extract(current_as_path):
                    stack.put(extracted)
    
    return document   
extract_text(file_path, txt=DefaultExtractor(), zip=ZipExtractor())